# HW3c - Cost-Sensitive Evaluation

See Canvas for details on how to complete and submit this assignment.

## Introduction

This is Part 3 of HW3. You'll apply cost-aware evaluation to an industrial maintenance problem, using class weights, threshold tuning, and `GridSearchCV` to optimize for business-relevant costs rather than accuracy. You saw this workflow in the Day 20 workshop on Bank Marketing; now you'll apply it independently on a different dataset and cost structure. The Reflection section at the end covers all three parts of HW3.

| Part | Topic | Release | Due | Points |
|------|-------|---------|-----|-------:|
| HW3a | Build & Evaluate Classifiers | Mar 17 | Mar 26 | 45 |
| HW3b | Paper: Cost-Sensitive Learning | Mar 17 | Mar 26 | 15 |
| HW3c | Cost-Sensitive Evaluation | Mar 24 | Mar 30 | 40 |
| | Total | | | 100 |

### Learning Objectives

By completing this part, you will demonstrate ability to:

- Use `GridSearchCV` for systematic hyperparameter search (reinforcement from HW3a)
- Apply class weights and threshold tuning to address class imbalance
- Evaluate models using business-relevant cost metrics, not just accuracy
- Connect cost-sensitive evaluation to the Elkan paper (HW3b)

### Time Estimate

This part should take ~2 hours to complete (plus ~15 minutes for the Reflection covering all of HW3).

### Generative AI Allowance

You may use GenAI tools for brainstorming, explanations, and code sketches if you disclose it, understand it, and validate it. Your submission must represent your own work and you are solely responsible for its correctness.

### Scoring

| Section | Points |
|---------|-------:|
| Setup and the Accuracy Problem | 5 |
| SVM and GridSearchCV | 10 |
| Class Weights and Cost Analysis | 8 |
| Threshold Tuning | 10 |
| Reflection (all of HW3) | 7 |
| **Total** | **40** |

## Background

You are a reliability engineer at a manufacturing plant. Your task is to predict machine failures before they happen, using sensor data from the production line. The catch: failures are rare (about 3.4% of observations), but missing one is expensive.

The AI4I 2020 Predictive Maintenance Dataset (UCI) contains 10,000 observations with sensor readings for an industrial machine. Features include air temperature, process temperature, rotational speed, torque, and tool wear. The binary target indicates whether the machine failed.

This is a synthetic dataset designed to reflect real industrial patterns. All features are physically meaningful - you can reason about *why* high torque at low speed might cause a failure.

## Setup

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

## Load Data and the Accuracy Problem (5 pts)

Load the dataset and examine the class distribution. Then fit a baseline logistic regression Pipeline and evaluate it.

In [ ]:
# Load the predictive maintenance dataset
url = "https://archive.ics.uci.edu/static/public/601/data.csv"
maint = pd.read_csv(url)

print(f"Shape: {maint.shape}")
print(f"Columns: {list(maint.columns)}")
maint.head()

In [ ]:
# Examine the target distribution
print(maint["Machine failure"].value_counts())
print(f"\nFailure rate: {maint['Machine failure'].mean():.1%}")

In [ ]:
# Prepare features and target
# Drop non-predictive columns: UID (row ID), Product ID, and the 5 individual
# failure mode columns (we're predicting overall Machine failure)
drop_cols = ["UID", "Product ID", "TWF", "HDF", "PWF", "OSF", "RNF", "Machine failure"]
X_maint = maint.drop(columns=drop_cols)
y_maint = maint["Machine failure"]

# The "Type" column (product quality: L, M, H) is categorical - encode it
X_maint = pd.get_dummies(X_maint, columns=["Type"], drop_first=True)

print(f"Features: {list(X_maint.columns)}")
print(f"Shape: {X_maint.shape}")

In [ ]:
# Train/test split with stratify - essential for rare events
X_tr, X_te, y_tr, y_te = train_test_split(
    X_maint, y_maint, test_size=0.3, random_state=42, stratify=y_maint
)

print(f"Training: {X_tr.shape[0]} ({y_tr.mean():.1%} failures)")
print(f"Test:     {X_te.shape[0]} ({y_te.mean():.1%} failures)")

In [ ]:
# Build, fit, and evaluate a baseline LogisticRegression Pipeline
# Same pattern as HW3a, using max_iter=1000
# Store in a variable called baseline (needed in later sections)
# Evaluate with: accuracy, classification_report, AND a normalized confusion matrix
# (use normalize="true" in ConfusionMatrixDisplay - the normalized version shows
# recall directly on the diagonal, which is more informative for imbalanced data)

# Your code here...

---

##### The Accuracy Problem

1. The baseline model achieves ~97% accuracy. Look at the normalized confusion matrix diagonal - what is the recall for failures (class 1)? Why is accuracy misleading here?
2. Why is `stratify=y_maint` even more important here than it was for the Steel Plates dataset in HW3a?

*Your answers here...*

---

## SVM and GridSearchCV (10 pts)

Support Vector Machines find the decision boundary that maximizes the margin (gap) between classes. The C parameter controls the tradeoff: large C fits the training data more tightly (low bias, high variance); small C allows more misclassifications for a wider margin (high bias, low variance).

Use `GridSearchCV` to find the best combination of C and kernel - same tool as HW3a's KNN tuning, but now searching over two parameters. Use `scoring="f1"` rather than `scoring="accuracy"`, given what you observed above. Use `StratifiedKFold(n_splits=5)` as the `cv` parameter to handle the class imbalance in each fold.

In [ ]:
# Use GridSearchCV to tune SVM hyperparameters
# Build a Pipeline: StandardScaler → SVC
# Search over C: [0.1, 1, 10, 100] and kernel: ["linear", "rbf"]
# Use cv=StratifiedKFold(n_splits=5) and scoring="f1"
# Note: this grid is 4×2 = 8 combinations × 5 folds = 40 fits.
# Should take ~30-60 seconds. Avoid adding extra C values or you may wait a while.

# Your code here... (store the fitted GridSearchCV in a variable called grid_search)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV F1:      {grid_search.best_score_:.3f}")

In [ ]:
# Evaluate the best SVM on the test set
# Use grid_search.predict() and print accuracy + classification_report
# Plot the confusion matrix with ConfusionMatrixDisplay.from_predictions()

# Your code here...

---

##### SVM Interpretation

1. What combination of C and kernel did `GridSearchCV` select? Why did we use `scoring="f1"` instead of `scoring="accuracy"`?
2. *Self-study*: in 2-3 sentences, explain what the *margin* is in an SVM and how the C parameter controls the bias-variance tradeoff. You may consult the 09b notebook, the Day 20 lecture, or external resources. Cite what you used.

*Your answers here...*

---

## Class Weights and Cost Analysis (8 pts)

Instead of manipulating data, we can tell the model that some mistakes are more expensive than others. `class_weight='balanced'` automatically upweights the minority class in proportion to its rarity. See the 11b notebook for examples.

In [ ]:
# Refit LogisticRegression with class_weight="balanced"
# Same Pipeline as baseline, but add class_weight="balanced" to LogisticRegression
# Store in a variable called balanced
balanced = Pipeline([
    ("scaler", StandardScaler()),
    # Add LogisticRegression with max_iter=1000 and class_weight="balanced"
])
balanced.fit(X_tr, y_tr)

In [ ]:
# Plot confusion matrices side by side: default vs balanced
# See 11b notebook for the subplot pattern
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_estimator(baseline, X_te, y_te, ax=axes[0])
axes[0].set_title("Default")
ConfusionMatrixDisplay.from_estimator(balanced, X_te, y_te, ax=axes[1])
axes[1].set_title("Balanced")
plt.tight_layout()
plt.show()

Now assign real costs and compute the financial impact of each model.

In [ ]:
# Cost assumptions:
# - False Negative (missed failure → unplanned downtime): $5,000
# - False Positive (unnecessary inspection): $200

cost_fn = 5000  # cost of missing a failure
cost_fp = 200   # cost of a false alarm

def compute_total_cost(y_true, y_pred, cost_fn, cost_fp):
    """Compute total misclassification cost."""
    cm = confusion_matrix(y_true, y_pred)
    # cm[1, 0] = false negatives, cm[0, 1] = false positives
    return cm[1, 0] * cost_fn + cm[0, 1] * cost_fp

# Use compute_total_cost to compare the default and balanced models
# Print the total cost for each and the savings

# Your code here...

---

##### Class Weights Interpretation

1. Compare the confusion matrices. What did `class_weight='balanced'` change? What did the model gain, and what did it trade away?
2. Which model has the lower total cost? Does this match which model has the higher accuracy? Why or why not?

*Your answers here...*

---

## Threshold Tuning (10 pts)

The default classification threshold is 0.5: if `predict_proba` returns P(failure) > 0.5, predict failure. But is 0.5 the right threshold for this problem? Let's look at what the model actually predicts.

In [ ]:
# Visualize the predicted probability distribution for the baseline model
# This shows how the model's confidence differs for actual failures vs non-failures
y_proba = baseline.predict_proba(X_te)[:, 1]  # P(failure) for each test sample

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(y_proba[y_te == 0], bins=50, alpha=0.5, label="No failure", color="steelblue")
ax.hist(y_proba[y_te == 1], bins=50, alpha=0.5, label="Failure", color="tomato")
ax.axvline(x=0.5, color="black", linestyle="--", label="Default threshold (0.5)")
ax.set_xlabel("Predicted P(failure)")
ax.set_ylabel("Count")
ax.set_title("Predicted Probability Distribution")
ax.legend()
plt.tight_layout()
plt.show()

---

##### What the Histogram Reveals

Look at where the red (failure) bars fall relative to the 0.5 threshold line. What does this tell you about why the default model catches so few failures? Where would you move the threshold to catch more?

*Your answer here...*

---

`TunedThresholdClassifierCV` (sklearn 1.5+) finds the optimal threshold automatically using cross-validation. It wraps an existing classifier and searches for the threshold that optimizes a given scoring function.

In [ ]:
from sklearn.model_selection import TunedThresholdClassifierCV

# Define a custom cost-sensitive scorer
# Lower total cost is better, so negate it (sklearn maximizes scores)
def cost_scorer(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    return -(cm[1, 0] * cost_fn + cm[0, 1] * cost_fp)

custom_scorer = make_scorer(cost_scorer)

In [ ]:
# Build a base LogReg Pipeline, then wrap it with TunedThresholdClassifierCV
# TunedThresholdClassifierCV takes: (estimator, scoring=..., cv=...)
# Use custom_scorer for scoring and StratifiedKFold(n_splits=5) for cv
# See sklearn docs for TunedThresholdClassifierCV
base_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])

# Wrap base_pipe with TunedThresholdClassifierCV, fit, and print the optimal threshold
# Store in a variable called tuned

# Your code here...

In [ ]:
# Compare all three approaches in a summary table
approaches = {
    "Default (t=0.5)": baseline.predict(X_te),
    "Balanced weights": balanced.predict(X_te),
    "Cost-optimized threshold": tuned.predict(X_te),
}

rows = []
for name, y_pred in approaches.items():
    cm = confusion_matrix(y_te, y_pred)
    rows.append({
        "Approach": name,
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "Recall": recall_score(y_te, y_pred),
        "F1": f1_score(y_te, y_pred),
        "FP": cm[0, 1],
        "FN": cm[1, 0],
        "Total Cost": cm[1, 0] * cost_fn + cm[0, 1] * cost_fp,
    })

cost_df = pd.DataFrame(rows)
print(cost_df.to_string(index=False))

---

##### Threshold Tuning Interpretation

1. What threshold did `TunedThresholdClassifierCV` find? Look back at your histogram - does this threshold make sense given the probability distribution you observed?
2. A manager reviews your results and asks: "Our accuracy dropped from 97% to [your number]. Why are we deploying a less accurate model?" Write 2-3 sentences explaining why the cost-optimized model is better despite lower accuracy.
3. Change `cost_fn` to 50000 (10x the original) and rerun the `TunedThresholdClassifierCV` cell and the comparison table cell. (`cost_scorer` looks up `cost_fn` at call time, so it picks up the new value automatically.) What happens to the optimal threshold? What happens to the total cost comparison? Explain why the threshold moved in the direction it did. *(Reset `cost_fn` back to 5000 and rerun those two cells when done.)*

*Your answers here...*

---

## Reflection (7 pts)

Address the following for all of HW3 (Parts a, b, and c). Concise bullets or short paragraphs are fine.

### 1. Key Takeaway

What part of this assignment most improved your understanding of classification or model evaluation? Include a concrete example.

---

##### Key Takeaway

*Your response here...*

---

### 2. GenAI Use

If you used GenAI tools (ChatGPT, Claude, etc.):
- What tool/model did you use?
- How did you use it (understanding concepts, debugging code, etc.)?
- How did you verify the output was correct?
- What worked well and what didn't?

If you didn't use GenAI, explain why and whether you plan to use it for future assignments.

---

##### GenAI Use

*Your response here...*

---

### 3. Feedback

- Approximately how much time did you spend on this assignment (per part)?
- What was the most difficult part?
- How would you improve this assignment?
- Anything else you want to share or ask?

---

##### Feedback

*Your response here...*

---